# 🎬 Agentado × DepthFlow — Cinematic Walkthrough Test
This notebook tests DepthFlow parallax rendering on a real estate listing photo.
**Free GPU runtime** — Runtime → Change runtime type → T4 GPU

In [ ]:
# 1. Install DepthFlow (takes ~2 min first time)
!pip install -q depthflow
!pip install -q opencv-python-headless pillow requests
print("✅ DepthFlow installed")

In [ ]:
# 2. Download a sample real estate listing photo
import requests, os
from PIL import Image

# Use a real listing photo (or replace URL with any photo)
# This is a high-res listing photo from one of our test listings
PHOTO_URL = "https://agentrocketman.com/agentado/api/img-proxy.php?u=https://cdn.realtor.ca/listing/TS638990698238200000/reb89/highres/2/29783482_2.jpg"

os.makedirs("input", exist_ok=True)
os.makedirs("output", exist_ok=True)

resp = requests.get(PHOTO_URL, stream=True)
with open("input/test_photo.jpg", "wb") as f:
    for chunk in resp.iter_content(8192):
        f.write(chunk)

img = Image.open("input/test_photo.jpg")
print(f"📸 Downloaded: {img.size[0]}×{img.size[1]} — {img.size[0]*img.size[1]/1e6:.1f}MP")
display(img.resize((640, int(640*img.size[1]/img.size[0]))))

In [ ]:
# 3. Run DepthFlow on the photo
# DepthFlow auto-detects depth and renders parallax video
# Options: --fps, --time, --resolution, --animation (orbit, dolly, etc)

!depthflow input/test_photo.jpg \
    --time 4 \
    --fps 30 \
    --resolution 1080x1920 \
    --animation dolly \
    --output output/cinematic_walkthrough.mp4 \
    --height 0.5 \
    --intensity 0.3

print("✅ Video rendered!")

In [ ]:
# 4. Preview the result
from IPython.display import Video
Video("output/cinematic_walkthrough.mp4", width=360, embed=True)

## 🔧 Next: Batch processing
Once this looks good, we'll wire it into Agentado's pipeline:
1. Listing photos → batch DepthFlow → parallax clips
2. Agentado adds: price intro, stats bar, agent end card
3. FFmpeg composes final video with music

## 🎛️ DepthFlow settings to tweak
- `--animation`: dolly, orbit, circle, horizontal, vertical
- `--intensity`: how much movement (0.0–1.0, try 0.2–0.5)
- `--height`: camera origin height (0.0=bottom, 1.0=top)
- `--time`: seconds per clip
- `--fps`: frame rate
- `--loop`: seamless looping video
- `--supersample 2`: renders at 2× resolution then downscales for anti-aliasing